# Goal: Demonstrate how to serve the AI as a REST API (FastAPI) instead of just a Streamlit dashboard.

Backend API Development (FastAPI)

Run this script as a standalone file (`uvicorn main:app --reload`) to serve predictions via HTTP.

In [ ]:
from fastapi import FastAPI, File, UploadFile
from ultralytics import YOLO
import cv2
import numpy as np
import io
from PIL import Image

app = FastAPI(title="Warehouse Safety API")
model = YOLO("../trained_models/yolo_worker_safety/weights/best.pt")

@app.post("/predict")
async def predict_image(file: UploadFile = File(...)):
    # Read the uploaded image
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")
    img_array = np.array(image)
    
    # Run Inference
    results = model(img_array)
    
    # Extract Bounding Boxes
    detections = []
    for box in results[0].boxes:
        detections.append({
            "class": int(box.cls[0]),
            "confidence": float(box.conf[0]),
            "bbox": box.xyxy[0].tolist()
        })
        
    return {"status": "success", "detections": detections}

# Note: In a real environment, you would run this via the terminal: 
# !pip install fastapi uvicorn python-multipart
# !uvicorn app:app --host 0.0.0.0 --port 8000